# California Housing Price Analysis

**Phân tích dữ liệu & So sánh thuật toán**

Dự án sử dụng **California Housing Dataset** để dự đoán giá nhà (đơn vị: $100,000s).

---

## Nội dung
1. **Exploratory Data Analysis (EDA)** - Thống kê mô tả, phân phối dữ liệu
2. **Correlation Analysis** - Ma trận tương quan giữa các đặc trưng
3. **Model Training & Comparison** - So sánh nhiều thuật toán ML
4. **Feature Importance** - Phân tích đặc trưng quan trọng
5. **Residual Analysis** - Kiểm tra phần dư
6. **Kết luận & Đề xuất**

In [ ]:
# ============================================================
# 1. IMPORT THU VIEN
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import os
from pathlib import Path

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

warnings.filterwarnings('ignore')

# Thiet lap style cho bieu do
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')

print('Import thu vien thanh cong!')

In [ ]:
# ============================================================
# 2. LOAD DU LIEU
# ============================================================
print('=' * 60)
print('LOAD DU LIEU CALIFORNIA HOUSING')
print('=' * 60)

# Load dataset
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

print(f'Da load du lieu thanh cong!')
print(f'Kich thuoc dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Danh sach features:')
for col in df.columns:
    print(f'  - {col}')
print(f'Target: MedHouseVal (Gia nha, don vi: $100,000s)')

In [ ]:
# ============================================================
# 3. EDA - EXPLORATORY DATA ANALYSIS
# ============================================================
print('=' * 60)
print('EDA - EXPLORATORY DATA ANALYSIS')
print('=' * 60)

# 3.1 Thong ke mo ta
print('\nThong ke mo ta:')
desc = df.describe().T
desc['missing'] = df.isnull().sum()
desc['missing_pct'] = (df.isnull().sum() / len(df)) * 100
display(desc.style.background_gradient(cmap='YlOrRd'))

# 3.2 Kiem tra gia tri thieu
print(f'\nTong so gia tri thieu: {df.isnull().sum().sum()}')

In [ ]:
# 3.3 Phan phoi cua target
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
axes[0].hist(df['MedHouseVal'], bins=50, edgecolor='black', alpha=0.7, color='#15803D')
axes[0].set_title('Distribution of MedHouseVal', fontsize=14, fontweight='bold')
axes[0].set_xlabel('MedHouseVal ($100,000s)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['MedHouseVal'].mean(), color='red', linestyle='--', label=f'Mean: {df["MedHouseVal"].mean():.2f}')
axes[0].axvline(df['MedHouseVal'].median(), color='blue', linestyle='--', label=f'Median: {df["MedHouseVal"].median():.2f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['MedHouseVal'], vert=False, patch_artist=True, boxprops=dict(facecolor='#22C55E'))
axes[1].set_title('Boxplot of MedHouseVal', fontsize=14, fontweight='bold')
axes[1].set_xlabel('MedHouseVal ($100,000s)')

# Summary stats
axes[2].axis('off')
mean_val = df['MedHouseVal'].mean()
std_val = df['MedHouseVal'].std()
min_val = df['MedHouseVal'].min()
q25 = df['MedHouseVal'].quantile(0.25)
med_val = df['MedHouseVal'].median()
q75 = df['MedHouseVal'].quantile(0.75)
max_val = df['MedHouseVal'].max()
skew_val = df['MedHouseVal'].skew()
stats_text = (
    f'Summary Statistics:\n'
    f'Count    : {len(df):,}\n'
    f'Mean     : {mean_val:.4f}\n'
    f'Std      : {std_val:.4f}\n'
    f'Min      : {min_val:.4f}\n'
    f'25%      : {q25:.4f}\n'
    f'50%      : {med_val:.4f}\n'
    f'75%      : {q75:.4f}\n'
    f'Max      : {max_val:.4f}\n'
    f'Skewness : {skew_val:.4f}'
)
axes[2].text(0.1, 0.5, stats_text, fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

print(f'Nhan xet: Du lieu target co phan phoi lech phai (skewness = {skew_val:.4f})')
print(f'Gia nha trung binh: ${mean_val * 100_000:,.0f}')

In [ ]:
# 3.4 Phan phoi cac features
features = [c for c in df.columns if c != 'MedHouseVal']
n_features = len(features)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = axes.flatten()

for i, feature in enumerate(features):
    axes[i].hist(df[feature], bins=40, edgecolor='black', alpha=0.7, color=sns.color_palette('viridis', n_features)[i])
    mean_f = df[feature].mean()
    std_f = df[feature].std()
    axes[i].set_title(f'{feature}\nMean: {mean_f:.4f} | Std: {std_f:.4f}', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

# An cac subplot thua
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle('Distribution of Features', fontsize=16, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# ============================================================
# 4. CORRELATION ANALYSIS
# ============================================================
print('=' * 60)
print('CORRELATION ANALYSIS')
print('=' * 60)

# 4.1 Correlation matrix
corr = df.corr(numeric_only=True)

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu',
    center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# 4.2 Top correlations with target
target_corr = corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)
print('\nTop features tuong quan voi MedHouseVal:')
for feat, val in target_corr.items():
    direction = '+' if val > 0 else '-'
    print(f'  {direction} {feat}: {val:.4f}')

# 4.3 Pairplot (lay mau de de nhin)
print('\nDang ve pairplot (sample 1000 rows)...')
df_sample = df.sample(1000, random_state=42)
sns.pairplot(df_sample, diag_kind='kde', plot_kws={'alpha': 0.5, 's': 10})
plt.suptitle('Pairplot of Features (Sample: 1000 rows)', fontsize=16, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# ============================================================
# 5. GEOGRAPHIC ANALYSIS
# ============================================================
print('=' * 60)
print('GEOGRAPHIC ANALYSIS')
print('=' * 60)

# Scatter plot theo vi tri dia ly
plt.figure(figsize=(14, 8))
scatter = plt.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseVal'], cmap='viridis',
    alpha=0.6, s=15, edgecolors='none'
)
plt.colorbar(scatter, label='MedHouseVal ($100,000s)')
plt.title('House Prices by Location', fontsize=16, fontweight='bold')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Nhan xet: Gia nha cao tap trung o khu vuc ven bien California (Longitude gan -122)')

In [ ]:
# ============================================================
# 6. MODEL TRAINING & COMPARISON
# ============================================================
print('=' * 60)
print('MODEL TRAINING & COMPARISON')
print('=' * 60)

# Prepare data
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]} samples')
print(f'Test size: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]} dac trung')

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=0.01),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5),
}

results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test = model.predict(X_test_scaled)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    
    # Cross-validation (5-fold)
    cv_rmse = -cross_val_score(model, X_train_scaled, y_train, cv=5, 
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_rmse_mean = cv_rmse.mean()
    
    results.append({
        'Model': name,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train R2': train_r2,
        'Test R2': test_r2,
        'Test MAE': test_mae,
        'CV RMSE': cv_rmse_mean,
    })
    
    trained_models[name] = model
    print(f'Done - Test RMSE: {test_rmse:.4f} | Test R2: {test_r2:.4f}')

# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Test R2', ascending=False).reset_index(drop=True)

print('\n' + '=' * 60)
print('KET QUA SO SANH CAC MODEL')
print('=' * 60)
display(results_df.style.background_gradient(cmap='Greens', subset=['Test R2', 'Test RMSE']))

In [ ]:
# ============================================================
# 7. VISUALIZE MODEL COMPARISON
# ============================================================
print('=' * 60)
print('VISUALIZE MODEL COMPARISON')
print('=' * 60)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 7.1 R2 Score Comparison
ax1 = axes[0]
bars1 = ax1.barh(results_df['Model'], results_df['Test R2'], color=sns.color_palette('viridis', len(results_df)))
ax1.set_xlabel('Test R2 Score', fontsize=12, fontweight='bold')
ax1.set_title('R2 Score Comparison', fontsize=14, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.axvline(x=0.8, color='red', linestyle='--', alpha=0.5, label='Excellent (0.8)')
ax1.axvline(x=0.6, color='orange', linestyle='--', alpha=0.5, label='Good (0.6)')
ax1.legend()
for bar, val in zip(bars1, results_df['Test R2']):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center')

# 7.2 RMSE Comparison
ax2 = axes[1]
bars2 = ax2.bar(results_df['Model'], results_df['Test RMSE'], color=sns.color_palette('magma', len(results_df)))
ax2.set_ylabel('Test RMSE', fontsize=12, fontweight='bold')
ax2.set_title('RMSE Comparison', fontsize=14, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
for bar, val in zip(bars2, results_df['Test RMSE']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}', ha='center', fontsize=10)

# 7.3 MAE Comparison
ax3 = axes[2]
bars3 = ax3.bar(results_df['Model'], results_df['Test MAE'], color=sns.color_palette('plasma', len(results_df)))
ax3.set_ylabel('Test MAE', fontsize=12, fontweight='bold')
ax3.set_title('MAE Comparison', fontsize=14, fontweight='bold')
ax3.tick_params(axis='x', rotation=45)
for bar, val in zip(bars3, results_df['Test MAE']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. BEST MODEL - DETAILED ANALYSIS
# ============================================================
# Tim model tot nhat dua tren Test R2
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]

print('=' * 60)
print(f'BEST MODEL: {best_model_name}')
print('=' * 60)

# Predict with best model
y_pred_best = best_model.predict(X_test_scaled)

# Residual
residuals = y_test - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 8.1 Actual vs Predicted
ax1 = axes[0, 0]
ax1.scatter(y_test, y_pred_best, alpha=0.5, s=15, c='#15803D')
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
ax1.set_xlabel('Actual Values', fontsize=12)
ax1.set_ylabel('Predicted Values', fontsize=12)
ax1.set_title(f'{best_model_name}: Actual vs Predicted', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 8.2 Residual Distribution
ax2 = axes[0, 1]
ax2.hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='#22C55E')
ax2.axvline(x=0, color='red', linestyle='--', lw=2, label='Zero Error')
ax2.set_xlabel('Residuals', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Residual Distribution', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 8.3 Residuals vs Predicted
ax3 = axes[1, 0]
ax3.scatter(y_pred_best, residuals, alpha=0.5, s=15, c='#D97706')
ax3.axhline(y=0, color='red', linestyle='--', lw=2)
ax3.set_xlabel('Predicted Values', fontsize=12)
ax3.set_ylabel('Residuals', fontsize=12)
ax3.set_title('Residuals vs Predicted', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 8.4 Q-Q Plot
ax4 = axes[1, 1]
from scipy import stats
stats.probplot(residuals, dist='norm', plot=ax4)
ax4.set_title('Q-Q Plot (Normality Check)', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

test_rmse_best = results_df.iloc[0]['Test RMSE']
test_r2_best = results_df.iloc[0]['Test R2']
test_mae_best = results_df.iloc[0]['Test MAE']
cv_best = results_df.iloc[0]['CV RMSE']
print(f'\nMetrics for {best_model_name}:')
print(f'  - Test RMSE: {test_rmse_best:.4f}')
print(f'  - Test R2:  {test_r2_best:.4f}')
print(f'  - Test MAE: {test_mae_best:.4f}')
print(f'  - CV RMSE:  {cv_best:.4f}')

In [ ]:
# ============================================================
# 9. FEATURE IMPORTANCE (cho tree-based models)
# ============================================================
print('=' * 60)
print('FEATURE IMPORTANCE ANALYSIS')
print('=' * 60)

feature_names = [c for c in df.columns if c != 'MedHouseVal']

# Lay feature importance tu cac model co ho tro
importance_models = {
    'Random Forest': trained_models.get('Random Forest'),
    'Decision Tree': trained_models.get('Decision Tree'),
    'Gradient Boosting': trained_models.get('Gradient Boosting'),
}

fig, axes = plt.subplots(1, len(importance_models), figsize=(20, 6))

for idx, (name, model) in enumerate(importance_models.items()):
    if model is not None and hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        ax = axes[idx]
        ax.bar(range(len(importances)), importances[indices], color=sns.color_palette('viridis', len(importances)))
        ax.set_xticks(range(len(importances)))
        ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
        ax.set_title(f'{name}\nFeature Importance', fontsize=14, fontweight='bold')
        ax.set_ylabel('Importance')
        ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Top feature chung
print('\nCac dac trung quan trong nhat (theo Random Forest):')
rf_model = trained_models.get('Random Forest')
if rf_model is not None and hasattr(rf_model, 'feature_importances_'):
    rf_imp = rf_model.feature_importances_
    sorted_idx = np.argsort(rf_imp)[::-1]
    for i in sorted_idx[:5]:
        print(f'  {i+1}. {feature_names[i]}: {rf_imp[i]:.4f}')

In [ ]:
# ============================================================
# 10. COEFFICIENTS ANALYSIS (cho Linear Models)
# ============================================================
print('=' * 60)
print('COEFFICIENTS ANALYSIS (Linear Models)')
print('=' * 60)

linear_models = {
    'Linear Regression': trained_models.get('Linear Regression'),
    'Ridge (L2)': trained_models.get('Ridge (L2)'),
    'Lasso (L1)': trained_models.get('Lasso (L1)'),
}

fig, axes = plt.subplots(1, len(linear_models), figsize=(20, 6))

for idx, (name, model) in enumerate(linear_models.items()):
    if model is not None and hasattr(model, 'coef_'):
        coefs = model.coef_
        colors = ['#1f77b4' if c >= 0 else '#d62728' for c in coefs]
        
        ax = axes[idx]
        ax.barh(feature_names, coefs, color=colors)
        ax.set_title(f'{name}\nCoefficients', fontsize=14, fontweight='bold')
        ax.set_xlabel('Coefficient Value')
        ax.axvline(x=0, color='black', linestyle='-', lw=1)
        ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print('\nGiai thich:')
print('  - Mau xanh: Tac dong tich cuc den gia nha')
print('  - Mau do: Tac dong tieu cuc den gia nha')
print('  - Do lon cua coefficient the hien muc do anh huong')

In [ ]:
# ============================================================
# 11. CROSS-VALIDATION COMPARISON
# ============================================================
print('=' * 60)
print('CROSS-VALIDATION COMPARISON')
print('=' * 60)

plt.figure(figsize=(12, 6))
bars = plt.bar(results_df['Model'], results_df['CV RMSE'], 
               color=sns.color_palette('viridis', len(results_df)))
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.ylabel('CV RMSE (5-fold)', fontsize=12, fontweight='bold')
plt.title('Cross-Validation RMSE Comparison', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, results_df['CV RMSE']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{val:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print('CV RMSE cang thap thi model cang on dinh')

In [ ]:
# ============================================================
# 12. HYPERPARAMETER TUNING (Random Forest)
# ============================================================
print('=' * 60)
print('HYPERPARAMETER TUNING - Random Forest')
print('=' * 60)

# Grid Search cho Random Forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5],
}

print('Dang tim kiem hyperparameters toi uu (co the mat vai phut)...')

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf, param_grid, cv=3, 
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV RMSE: {-grid_search.best_score_:.4f}')

# Evaluate best model
best_rf = grid_search.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_scaled)
test_rmse_rf_tuned = np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned))
test_r2_rf_tuned = r2_score(y_test, y_pred_rf_tuned)

print(f'Tuned Random Forest Test RMSE: {test_rmse_rf_tuned:.4f}')
print(f'Tuned Random Forest Test R2: {test_r2_rf_tuned:.4f}')

# So sanh truoc va sau tuning
print('\nSo sanh truoc va sau tuning:')
rf_before = results_df[results_df['Model'] == 'Random Forest'].iloc[0]
print(f'  - Truoc tuning: RMSE={rf_before["Test RMSE"]:.4f}, R2={rf_before["Test R2"]:.4f}')
print(f'  - Sau tuning:   RMSE={test_rmse_rf_tuned:.4f}, R2={test_r2_rf_tuned:.4f}')
improvement_rmse = rf_before['Test RMSE'] - test_rmse_rf_tuned
improvement_r2 = test_r2_rf_tuned - rf_before['Test R2']
print(f'  - Cai thien:    RMSE giam {improvement_rmse:.4f}, R2 tang {improvement_r2:.4f}')

In [ ]:
# ============================================================
# 13. SAVE METRICS & FINAL SUMMARY
# ============================================================
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)

# Save comparison results to JSON
summary = {
    'dataset': {
        'name': 'California Housing',
        'samples': len(df),
        'features': len(feature_names),
        'target': 'MedHouseVal ($100,000s)'
    },
    'train_test_split': {
        'train_size': int(X_train.shape[0]),
        'test_size': int(X_test.shape[0]),
        'test_ratio': 0.2
    },
    'models_comparison': results,
    'best_model': {
        'name': best_model_name,
        'test_rmse': float(results_df.iloc[0]['Test RMSE']),
        'test_r2': float(results_df.iloc[0]['Test R2']),
        'test_mae': float(results_df.iloc[0]['Test MAE']),
    }
}

# Add tuned model info
summary['tuned_model'] = {
    'name': 'Random Forest (Tuned)',
    'best_params': grid_search.best_params_,
    'test_rmse': float(test_rmse_rf_tuned),
    'test_r2': float(test_r2_rf_tuned),
}

# Save to file
output_path = Path.cwd() / 'analysis_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'Ket qua duoc luu vao: {output_path}')

print('\n' + '=' * 60)
print('KET LUAN')
print('=' * 60)
print(f'''
Model tot nhat: {best_model_name}
  - Test RMSE: {results_df.iloc[0]["Test RMSE"]:.4f}
  - Test R2:  {results_df.iloc[0]["Test R2"]:.4f}
  - Test MAE: {results_df.iloc[0]["Test MAE"]:.4f}

Dac trung quan trong nhat: MedInc (Thu nhap trung binh)
Khu vuc co gia nha cao: Ven bien California
''')

In [ ]:
# ============================================================
# 14. SO SANH VOI MODEL HIEN TAI
# ============================================================
print('=' * 60)
print('SO SANH VOI MODEL HIEN TAI (tu train_model.py)')
print('=' * 60)

# Load current model metrics
metrics_path = Path.cwd() / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path, 'r') as f:
        current_metrics = json.load(f)
    
    print(f'\nModel hien tai (Linear Regression tu train_model.py):')
    print(f'  - RMSE: {current_metrics["rmse"]:.4f}')
    print(f'  - R2:  {current_metrics["r2"]:.4f}')
    print(f'  - Train samples: {current_metrics["n_train"]:,}')
    print(f'  - Test samples: {current_metrics["n_test"]:,}')
    
    print(f'\nModel tot nhat tu Notebook ({best_model_name}):')
    print(f'  - RMSE: {results_df.iloc[0]["Test RMSE"]:.4f}')
    print(f'  - R2:  {results_df.iloc[0]["Test R2"]:.4f}')
    
    improvement = current_metrics['rmse'] - results_df.iloc[0]['Test RMSE']
    if improvement > 0:
        print(f'\nModel {best_model_name} cai thien RMSE: {improvement:.4f} so voi Linear Regression!')
    else:
        print(f'\nLinear Regression hien tai co RMSE tot hon {-improvement:.4f}')
else:
    print('Khong tim thay metrics.json. Chay train_model.py de tao.')

print('\n' + '=' * 60)
print('HOAN THANH PHAN TICH!')
print('=' * 60)